In [2]:
from transformers import AutoProcessor, AutoModel
import torch
from PIL import Image
import cairosvg
import os
import gc

class SVGMetricEvaluator:
    def __init__(self, model_name="google/siglip-so400m-patch14-384", device=None):
        # Initialize the device and model
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu") if device is None else device
        print(f"Using device: {self.device}")
        
        # Load the model and processor
        self.model = AutoModel.from_pretrained(model_name)
        self.processor = AutoProcessor.from_pretrained(model_name)
    
    def svg_metric(self, prompt, svg):
        try:
            # Convert SVG to PNG
            cairosvg.svg2png(svg, write_to="./tmp/temp.png")
            
            # Open and process the image
            image = Image.open('./tmp/temp.png').convert("RGB")
            texts = ["SVG illustration of " + prompt]
            inputs = self.processor(text=texts, images=image, padding="max_length", return_tensors="pt")
            
            # Inference without gradient tracking
            with torch.no_grad():
                outputs = self.model(**inputs)
            
            logits_per_image = outputs.logits_per_image
            probs = torch.sigmoid(logits_per_image)
            
            # Clean up temporary PNG file
            os.remove('./tmp/temp.png')
            
            return probs[0][0].item()
        
        except Exception as e:
            print(f"An error occurred: {e}")
            return None


In [4]:
import pandas as pd
df=pd.read_csv('./drawing-with-llms/svg_score_test.csv',header=[0])
df=df[['description','svg']]
print(df.shape)

(76, 2)


In [3]:
import pandas as pd
from tqdm import tqdm
import time
tqdm.pandas()
start_time = time.time()

evaluator = SVGMetricEvaluator()

df['sl_score'] = df.progress_apply(lambda row: evaluator.svg_metric(row['description'], row['svg']), axis=1)

sl_score = df['sl_score'].mean()

# Measure the elapsed time
elapsed_time = time.time() - start_time
print(f"Time taken for processing: {elapsed_time:.2f} seconds")

# Print the average score
print(f"Average SL score: {sl_score}")


Using device: cuda


100%|███████████████████████████████████████████| 76/76 [02:00<00:00,  1.59s/it]

Time taken for processing: 125.04 seconds
Average SL score: 0.9176209670932669


In [11]:
from transformers import AutoProcessor, AutoModel
import torch
from PIL import Image
import cairosvg
import os
import gc
import io
from concurrent.futures import ThreadPoolExecutor, as_completed

class SVGMetricEvaluator:
    def __init__(self, model_name="google/siglip-so400m-patch14-384", device=None, max_workers=4):
        # Initialize the device and model
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu") if device is None else device
        print(f"Using device: {self.device}")
        
        # Load the model and processor
        self.model = AutoModel.from_pretrained(model_name)
        self.processor = AutoProcessor.from_pretrained(model_name)
        
        # Initialize the ThreadPoolExecutor
        self.executor = ThreadPoolExecutor(max_workers=max_workers)

    def svg_metric(self, prompt, svg):
        try:

            png_data = cairosvg.svg2png(bytestring=svg.encode('utf-8'))  # Convert SVG string to PNG data

            image = Image.open(io.BytesIO(png_data)).convert("RGB")
            
            texts = ["SVG illustration of " + prompt]
            inputs = self.processor(text=texts, images=image, padding="max_length", return_tensors="pt")
            
            # Inference without gradient tracking
            with torch.no_grad():
                outputs = self.model(**inputs)
            
            logits_per_image = outputs.logits_per_image
            probs = torch.sigmoid(logits_per_image)
            
           
            return probs[0][0].item()
        
        except Exception as e:
            print(f"An error occurred: {e}")
            return None

    def evaluate_concurrent(self, prompts_svgs):
        futures = []
        
        # Submit tasks to the ThreadPoolExecutor
        for prompt, svg in prompts_svgs:
            future = self.executor.submit(self.svg_metric, prompt, svg)
            futures.append(future)

        results = []
        
        # Gather results as tasks complete
        for future in as_completed(futures):
            result = future.result()
            results.append(result)
        
        return results

